# Stal Rzeszów – tracking (Colab)
Cała logika siedzi w pakiecie `stal_tracking`. Tu tylko: instalacja, config, uruchomienie.
Runtime → GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = 'https://github.com/TWOJ_LOGIN/stal-tracking.git'  # <- podmień
!test -d /content/stal-tracking || git clone -q $REPO /content/stal-tracking
!cd /content/stal-tracking && git pull -q && pip install -q -e '.[rfdetr,weights]'

In [ ]:
from stal_tracking import Config

cfg = Config.from_yaml('/content/stal-tracking/configs/stal_5_minut.yaml')
# szybkie eksperymenty bez edycji pliku, np.: Config.from_yaml(..., merge_thresh=0.9)
cfg

In [ ]:
from stal_tracking.weights import ensure_reid_weights
ensure_reid_weights(cfg.weights_dir)  # tylko za pierwszym razem pobiera i konwertuje

In [ ]:
from stal_tracking.detectors import make_detector
from stal_tracking.tracking import diagnose

diagnose(cfg, make_detector(cfg.detector, cfg.det_conf))

In [ ]:
from stal_tracking.pipeline import run

# Detekcje i tracki online są cache'owane na Drive – zmiana merge_thresh itp. nie odpala RF-DETR ponownie.
final = run(cfg)

In [ ]:
import cv2, matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(cfg.out('_arcsort.mp4')))
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for ax, idx in zip(axes, [n // 4, n // 2, 3 * n // 4]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, fr = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
    ax.set_title(f'klatka {idx}'); ax.axis('off')
cap.release(); plt.tight_layout(); plt.show()